In [2]:
def to_signed_28bit(n):
    # Ensure we are only looking at the bottom 28 bits
    n = n & 0xFFFFFFF 
    
    # If the 28th bit is set (0x8000000)
    if n & 0x8000000:
        # Subtract 2^28 to get the signed value
        return n - 0x10000000
    
    return n

# Examples:
print(to_signed_28bit(0x0000001))  # Result: 1
print(to_signed_28bit(0x8000000))  # Result: -134217728 (Min value)
print(to_signed_28bit(0xFFFFFFF))  # Result: -1 (Max negative)

1
-134217728
-1


In [11]:
import math

def float_to_custom_format(val):
    if val == 0:
        return 0, 0, 0

    # 1. Decompose the float
    # mantissa is [0.5, 1), exp is the power of 2
    mant, exp = math.frexp(val)

    # 2. Handle the 10-bit Offset Exponent
    # Assuming a standard mid-point bias (2^(10-1) - 1) = 511
    bias = 512
    offset_exponent = exp + bias
    
    # Clamp to ensure it fits in 10 bits (0 to 1023)
    offset_exponent = max(0, min(1023, offset_exponent))

    # 3. Handle the 28-bit Signed Mantissa
    # We scale the [0.5, 1) mantissa to a 28-bit integer range.
    # Max value for 28-bit signed is 2^27 - 1
    mant_scaled = int(mant * (1 << 27))
    
    # Ensure it stays within 28-bit signed bounds
    mask_28 = (1 << 28) - 1
    mant_28 = mant_scaled & mask_28

    # 4. Split Mantissa into two 16-bit words (Right Aligned)
    # Word 2 gets the lower 16 bits
    # Word 1 gets the remaining 12 bits
    word2 = mant_28 & 0xFFFF
    word1 = (mant_28 >> 16) & 0x0FFF  # Only 12 bits used here

    return offset_exponent, word1, word2

# Example Usage:
v = 16.333333333333333333
exp_w, m_hi, m_lo = float_to_custom_format(v)

print(f"Value: {v}")
print(f"Word 0 (Exp): {oct(exp_w)}")
print(f"Word 1 (Mant Hi): {oct(m_hi)}")
print(f"Word 2 (Mant Lo): {oct(m_lo)}")

Value: 16.333333333333332
Word 0 (Exp): 0o1005
Word 1 (Mant Hi): 0o2025
Word 2 (Mant Lo): 0o52525


In [11]:
import math

def custom_format_to_float(word0, word1, word2, bias=512):
    """
    word0: 10-bit offset exponent
    word1: Upper 12 bits of 28-bit mantissa
    word2: Lower 16 bits of 28-bit mantissa
    """
    # 1. Reassemble the 28-bit mantissa
    mant_28 = ((word1&0xFFF) << 16) | (word2&0xFFFF)
    
    # 2. Handle the sign (Two's complement for 28 bits)
    # Check if the 28th bit (index 27) is set
    if mant_28 & (1 << 27):
        # Subtract 2^28 to get the negative value
        mant_val = mant_28 - (1 << 28)
    else:
        mant_val = mant_28
        
    # 3. Handle zero case
    if word0 == 0 and mant_val == 0:
        return 0.0

    # 4. Reconstruct the float
    # The exponent we stored was: exp + bias
    # The mantissa was scaled by: 2^27
    # So: float = mant_val * 2^(word0 - bias - 27)
    return math.ldexp(mant_val, word0 - bias - 27)

In [14]:
recovered = custom_format_to_float(0o0777, 0o4000, 0o00000)
print(recovered)

-0.5


In [1]:
import math
val=16.333333333333
mant,exp=math.frexp(val)
exp=max(0,min(exp+512,1023))
mant28=int(mant*(1<<27))
ml=mant28&0xFFFF
mh=(mant28>>16)&0x0FFF
print(f'exp:{oct(exp)} mh:{oct(mh)} ml:{oct(ml)}')

exp:0o1005 mh:0o2025 ml:0o52525


In [7]:
exp=0o1005;mh=0o2000;ml=0o00000

exp=exp-512
mant=((mh&0xFFF)<<16)|(ml&0xFFFF)
if mant&1<<27:
    mant=mant-(1<<28)
mant=mant*(2**-27)
val=(2**exp)*mant
print(val,mant,exp)

16.0 0.5 5
